# Week 1 (end) — Honest Classical Baseline

**Goal:** get the number that replaces the old, fake 100%. Train TF-IDF + Logistic Regression and TF-IDF + Linear SVM on the leakage-safe split from `03_split.ipynb`, and report them on three sets:

| Set | Question it answers |
|---|---|
| `test` | How good is the model on essays like the ones it trained on? |
| `heldout_prompt` | …on topics it never saw? |
| `heldout_generator` | …on AI models (Claude, Falcon-180B) it never saw? |

Two rules keep the numbers honest:
- **Everything is fitted on `train`**, including the TF-IDF vocabulary. **Every choice is made on `val`**. Test and the held-out slices are scored once, at the end.
- **Reference models first.** A score only means something next to what you get for free. The data card lists two free signals, essay length and topic, so we measure how far each one gets on its own.

This is also the template every later model (BiLSTM, BERT, …) is compared against.

In [1]:
import sys
sys.path.append('..')

import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)

from src.data import load_splits

SEED = 42
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 60)

## Step 1 — Load the splits

Always through `load_splits`, which strips the leading/trailing whitespace that leaked the label (see `03_split`).

In [2]:
data = load_splits()
train, val, test = (data[data['split'] == s] for s in ['train', 'val', 'test'])
ho_prompt = data[data['split'] == 'heldout_prompt']
ho_gen = data[data['split'] == 'heldout_generator']

for name, d in [('train', train), ('val', val), ('test', test),
                ('heldout_prompt', ho_prompt), ('heldout_generator', ho_gen)]:
    print(f"{name:18s} {len(d):6d} essays, AI share {d['label'].mean():.3f}")

train               28006 essays, AI share 0.346
val                  3501 essays, AI share 0.346
test                 3501 essays, AI share 0.346
heldout_prompt       7135 essays, AI share 0.374
heldout_generator    2721 essays, AI share 1.000


## Step 2 — How we score a model

Accuracy alone is misleading here: train/val/test are 65% human, so a model that always says "human" scores 65%. We report:

- **Precision** (of essays flagged AI, how many really are). Low precision means innocent students get accused.
- **Recall / detection rate** (of AI essays, how many get flagged).
- **F1**, the balance of the two.
- **ROC-AUC**: the chance that a random AI essay gets a higher "AI score" than a random human essay. It doesn't depend on the 0.5 threshold or on the class ratio, so it's the main number for comparing models. 0.5 = coin flip, 1.0 = perfect ranking.

**The held-out-generator slice is AI-only**, so precision and AUC can't be computed on it alone (there are no human essays to confuse). We report its **detection rate**, plus an **AUC paired with the human essays of `test`**: "can the model rank unseen-family AI essays above human essays?"

`evaluate` takes a fitted model's scores and threshold so the same code serves every model.

In [3]:
def metrics(y, score, threshold):
    pred = (score >= threshold).astype(int)
    return {'acc': accuracy_score(y, pred),
            'precision': precision_score(y, pred, zero_division=0),
            'recall': recall_score(y, pred),
            'f1': f1_score(y, pred),
            'auc': roc_auc_score(y, score)}

def evaluate(score_fn, threshold, name):
    # score_fn(df) -> 1-D array, higher = more likely AI
    rows = {}
    for split_name, d in [('test', test), ('heldout_prompt', ho_prompt)]:
        rows.update({f'{split_name}_{k}': v for k, v in metrics(d['label'], score_fn(d), threshold).items()})
    gen_score = score_fn(ho_gen)
    test_humans = test[test['label'] == 0]
    paired_y = np.r_[np.zeros(len(test_humans)), np.ones(len(ho_gen))]
    paired_score = np.r_[score_fn(test_humans), gen_score]
    rows['heldout_gen_detection'] = (gen_score >= threshold).mean()
    rows['heldout_gen_auc'] = roc_auc_score(paired_y, paired_score)
    return pd.Series(rows, name=name)

results = []

## Step 3 — Reference models (what you get for free)

**3a. Majority class:** always say "human". Every score is the same constant, so AUC is exactly 0.5.

In [4]:
results.append(evaluate(lambda d: np.zeros(len(d)), threshold=0.5, name='always human'))
results[-1].round(3).to_frame().T

,test_acc,test_precision,test_recall,test_f1,test_auc,heldout_prompt_acc,heldout_prompt_precision,heldout_prompt_recall,heldout_prompt_f1,heldout_prompt_auc,heldout_gen_detection,heldout_gen_auc
always human,0.654,0.0,0.0,0.0,0.5,0.626,0.0,0.0,0.0,0.5,0.0,0.5


**3b. Length only.** A decision tree (depth 3) that sees *only* the word count. A tree rather than logistic regression because the length signal isn't one-directional: AI essays sit in a narrow 250–400-word band, while human essays are both shorter and longer. A depth-3 tree can learn "inside the band → AI" with at most 8 length ranges.

In [5]:
length_tree = DecisionTreeClassifier(max_depth=3, random_state=SEED)
length_tree.fit(train[['n_words']], train['label'])
print("val AUC:", round(roc_auc_score(val['label'], length_tree.predict_proba(val[['n_words']])[:, 1]), 3))

results.append(evaluate(lambda d: length_tree.predict_proba(d[['n_words']])[:, 1], 0.5, 'length only (tree)'))
results[-1].round(3).to_frame().T

val AUC: 0.691


,test_acc,test_precision,test_recall,test_f1,test_auc,heldout_prompt_acc,heldout_prompt_precision,heldout_prompt_recall,heldout_prompt_f1,heldout_prompt_auc,heldout_gen_detection,heldout_gen_auc
length only (tree),0.673,1.0,0.055,0.103,0.719,0.631,0.956,0.016,0.032,0.726,0.002,0.75


**3c. Prompt only.** Logistic regression on a one-hot encoding of the prompt. This is just "predict this prompt's AI share". It knows nothing about the essay itself.

On `heldout_prompt` every prompt is unseen, so `handle_unknown='ignore'` turns it into an all-zero vector and every essay gets the same score. By construction this model *can't* work there, and that's the point: the slice removes the topic shortcut.

In [6]:
prompt_ohe = OneHotEncoder(handle_unknown='ignore')
prompt_lr = LogisticRegression(max_iter=1000)
prompt_lr.fit(prompt_ohe.fit_transform(train[['prompt_name']]), train['label'])
prompt_score = lambda d: prompt_lr.predict_proba(prompt_ohe.transform(d[['prompt_name']]))[:, 1]
print("val AUC:", round(roc_auc_score(val['label'], prompt_score(val)), 3))

results.append(evaluate(prompt_score, 0.5, 'prompt only'))
results[-1].round(3).to_frame().T

val AUC: 0.791


,test_acc,test_precision,test_recall,test_f1,test_auc,heldout_prompt_acc,heldout_prompt_precision,heldout_prompt_recall,heldout_prompt_f1,heldout_prompt_auc,heldout_gen_detection,heldout_gen_auc
prompt only,0.73,0.625,0.544,0.582,0.763,0.626,0.0,0.0,0.0,0.5,0.204,0.513


## Step 4 — TF-IDF features

**TF-IDF** turns each essay into a long sparse vector with one dimension per word or word pair (n-gram):
- **TF** (term frequency): how often the term appears in *this* essay. With `sublinear_tf=True` we use 1 + log(count), so a word used 20 times doesn't count 20× more than one used once.
- **IDF** (inverse document frequency): down-weights terms that appear in almost every essay ("the", "and"), which carry little information.

Settings, and why:
- `ngram_range=(1, 2)`: single words *and* word pairs. Pairs capture phrasing ("in conclusion", "additionally ,"), which is where AI style shows.
- `min_df=3`: drop terms seen in fewer than 3 essays (typos, names); they can't generalise.
- `max_features=200_000`: keep the 200k most frequent terms, which bounds memory.

**The vectorizer is fitted on `train` only.** If it saw val/test, their vocabulary and word frequencies would leak into the features. Val/test/held-out essays are only *transformed* with the train vocabulary.

**Sanity check first** (as in `02`): fit on 2,000 training essays and score val. That checks the pipeline end to end in seconds, and the number should already be well above 0.5.

In [7]:
small = train.sample(2000, random_state=SEED)
vec_small = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True)
X_small = vec_small.fit_transform(small['text'])
lr_small = LogisticRegression(max_iter=1000).fit(X_small, small['label'])
print("vocab size:", X_small.shape[1])
print("val AUC (2,000 training essays):", round(roc_auc_score(val['label'], lr_small.predict_proba(vec_small.transform(val['text']))[:, 1]), 3))

vocab size: 42478


val AUC (2,000 training essays): 0.997


The pipeline works. Now the full training set.

In [8]:
t0 = time.time()
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200_000, sublinear_tf=True)
X_train = vec.fit_transform(train['text'])
X = {name: vec.transform(d['text']) for name, d in
     [('val', val), ('test', test), ('heldout_prompt', ho_prompt), ('heldout_generator', ho_gen)]}
print(f"X_train: {X_train.shape[0]} essays x {X_train.shape[1]} terms  ({time.time() - t0:.0f}s)")
print(f"non-zero entries per essay (mean): {X_train.getnnz(axis=1).mean():.0f}")

X_train: 28006 essays x 200000 terms  (33s)
non-zero entries per essay (mean): 419


## Step 5 — Train and tune on `val`

Both models learn one weight per term and add them up into a score. They differ in the loss they minimise:
- **Logistic Regression** minimises log-loss and outputs a probability (threshold 0.5).
- **Linear SVM** maximises the margin between classes and outputs a signed distance (threshold 0).

**`C`** controls regularisation: small `C` keeps weights small (simpler model, less overfitting); large `C` lets the model fit the training data harder. We try a few values and keep the one with the best **val** AUC. Test is not looked at.

The score functions take a dataframe and look up its pre-computed TF-IDF matrix by split name, so `evaluate` can stay generic.

In [9]:
def tfidf_rows(d):
    # pre-computed matrix for a split, or a subset of test (the paired-AUC humans)
    split = d['split'].iloc[0]
    if split == 'test' and len(d) != len(test):
        return X['test'][test.index.get_indexer(d.index)]
    return X[split]

tuning = []
for model_name, make in [('logreg', lambda C: LogisticRegression(C=C, max_iter=2000)),
                         ('linear_svm', lambda C: LinearSVC(C=C))]:
    for C in [0.1, 1, 10] if model_name == 'logreg' else [0.01, 0.1, 1]:
        t0 = time.time()
        m = make(C).fit(X_train, train['label'])
        s = m.decision_function(X['val'])
        tuning.append({'model': model_name, 'C': C, 'val_auc': roc_auc_score(val['label'], s),
                       'val_f1': f1_score(val['label'], (s >= 0).astype(int)), 'fit_s': round(time.time() - t0, 1)})
tuning = pd.DataFrame(tuning).round(4)
tuning

,model,C,val_auc,val_f1,fit_s
0,logreg,0.10,0.9978,0.9584,6.0
1,logreg,1.00,0.9994,0.9845,7.1
2,logreg,10.00,0.9997,0.9887,4.8
3,linear_svm,0.01,0.9981,0.9615,1.1
4,linear_svm,0.10,0.9996,0.9879,0.8
5,linear_svm,1.00,0.9999,0.9942,1.0


(`decision_function` is used for both models during tuning: for logistic regression it's the log-odds, and log-odds ≥ 0 ⇔ probability ≥ 0.5, so thresholding at 0 is the same decision.)

Refit the best `C` per model and score once on the final sets.

In [10]:
best = tuning.loc[tuning.groupby('model')['val_auc'].idxmax()].set_index('model')
print(best[['C', 'val_auc']])

logreg = LogisticRegression(C=best.loc['logreg', 'C'], max_iter=2000).fit(X_train, train['label'])
svm = LinearSVC(C=best.loc['linear_svm', 'C']).fit(X_train, train['label'])

results.append(evaluate(lambda d: logreg.predict_proba(tfidf_rows(d))[:, 1], 0.5,
                        f"TF-IDF + LogReg (C={best.loc['logreg', 'C']:g})"))
results.append(evaluate(lambda d: svm.decision_function(tfidf_rows(d)), 0.0,
                        f"TF-IDF + LinearSVM (C={best.loc['linear_svm', 'C']:g})"))

               C  val_auc
model                    
linear_svm   1.0   0.9999
logreg      10.0   0.9997


## Step 6 — Results

One row per model. Read across a row to see how a model holds up on unseen topics and unseen generators.

In [11]:
table = pd.DataFrame(results)
cols = ['test_acc', 'test_precision', 'test_recall', 'test_f1', 'test_auc',
        'heldout_prompt_f1', 'heldout_prompt_auc', 'heldout_gen_detection', 'heldout_gen_auc']
table[cols].round(3)

,test_acc,test_precision,test_recall,test_f1,test_auc,heldout_prompt_f1,heldout_prompt_auc,heldout_gen_detection,heldout_gen_auc
always human,0.654,0.000,0.000,0.000,0.500,0.000,0.500,0.000,0.500
length only (tree),0.673,1.000,0.055,0.103,0.719,0.032,0.726,0.002,0.750
prompt only,0.730,0.625,0.544,0.582,0.763,0.000,0.500,0.204,0.513
TF-IDF + LogReg (C=10),0.994,0.997,0.984,0.991,1.000,0.913,0.990,0.897,1.000
TF-IDF + LinearSVM (C=1),0.997,1.000,0.990,0.995,1.000,0.973,0.997,0.909,1.000


In [12]:
print("Confusion matrix, TF-IDF + LogReg on test (rows = true, cols = predicted; 0 = human, 1 = AI)")
pred = logreg.predict(X['test'])
print(pd.DataFrame(confusion_matrix(test['label'], pred), index=['true human', 'true AI'], columns=['pred human', 'pred AI']))

Confusion matrix, TF-IDF + LogReg on test (rows = true, cols = predicted; 0 = human, 1 = AI)
            pred human  pred AI
true human        2288        3
true AI             19     1191


## Step 7 — Where does it fail?

A single number hides *which* essays the model gets wrong. Break the test and held-out errors down by **source**: for AI sources the rate that matters is how many are caught (recall); for human sources, how many are wrongly flagged (false-positive rate).

In [13]:
def by_source(d, score):
    pred = score >= 0.5
    out = pd.DataFrame({'source': d['source'], 'label': d['label'], 'flagged_ai': pred})
    g = out.groupby(['label', 'source'])['flagged_ai'].agg(['size', 'mean']).rename(columns={'size': 'essays', 'mean': '% flagged AI'})
    g['% flagged AI'] = (g['% flagged AI'] * 100).round(1)
    return g

lr_score = lambda d: logreg.predict_proba(tfidf_rows(d))[:, 1]
print("TEST (label 0 rows: false-positive rate; label 1 rows: detection rate)")
print(by_source(test, lr_score(test)))
print("\nHELD-OUT GENERATORS")
print(by_source(ho_gen, lr_score(ho_gen)))
print("\nHELD-OUT PROMPTS, by prompt")
hp = ho_prompt.assign(flagged=lr_score(ho_prompt) >= 0.5)
print(hp.groupby(['prompt_name', 'label'])['flagged'].mean().mul(100).round(1).unstack().rename(columns={0: '% humans flagged', 1: '% AI caught'}))

TEST (label 0 rows: false-positive rate; label 1 rows: detection rate)
                                          essays  % flagged AI
label source                                                  
0     persuade_corpus                       2229           0.1
      train_essays                            62           0.0
1     NousResearch/Llama-2-7b-chat-hf         14         100.0
      chat_gpt_moth                          236         100.0
      cohere-command                          29          93.1
      kingki19_palm                           82         100.0
      llama2_chat                            198         100.0
      llama_70b_v1                           102          98.0
      mistral7binstruct_v1                   233         100.0
      mistral7binstruct_v2                   232          93.5
      mistralai/Mistral-7B-Instruct-v0.1      21         100.0
      palm-text-bison1                        27         100.0
      radek_500                               2

## Step 8 — What did it learn?

Logistic regression is a weighted sum of terms, so its largest weights *are* its explanation: positive pushes towards AI, negative towards human. Check whether these look like writing style or like artifacts (placeholders, formatting).

In [14]:
names = vec.get_feature_names_out()
coef = logreg.coef_[0]
top = 25
print("Most AI-indicative terms:\n ", ', '.join(names[np.argsort(coef)[-top:][::-1]]))
print("\nMost human-indicative terms:\n ", ', '.join(names[np.argsort(coef)[:top]]))

Most AI-indicative terms:
  re, it like, important, it not, important to, additionally, essay, conclusion, potential, you re, in conclusion, it important, super, success, lead to, provide, goals, and stuff, hey, this essay, benefits, impact, significant, and, is important

Most human-indicative terms:
  because, would, very, you, will, many, car, venus, most, home, then, the students, do, at, people, go, could, no, cars, nasa, your, all, earth, only, if


Read a few real mistakes on `test`: the human essays the model is *most* sure are AI (false positives, the costly error), and the AI essays it is most sure are human.

In [15]:
t = test.assign(p_ai=lr_score(test))
fp = t[(t['label'] == 0)].nlargest(3, 'p_ai')
fn = t[(t['label'] == 1)].nsmallest(3, 'p_ai')
for title, rows in [("HUMAN essays scored most AI-like (false positives)", fp),
                    ("AI essays scored most human-like (false negatives)", fn)]:
    print(f"===== {title} =====")
    for _, r in rows.iterrows():
        print(f"[p_ai={r['p_ai']:.2f} | {r['source']} | {r['prompt_name']} | {r['n_words']} words]")
        print(r['text'][:450].replace('\n', ' '), '...\n')

===== HUMAN essays scored most AI-like (false positives) =====
[p_ai=0.62 | persuade_corpus | Phones and driving | 538 words]
It is hypothesized that despite an effective way of communication, using cell phones during driving results in distracting attention of driver increasing road safety concerns. Cell phones were introduced in the United States market in the decade of 1980 and their usage instantly grew during next two decades. The trend of using cell phones has increased throughout the world. One of the primary reasons for the gigantic growth of cell phones is that ...

[p_ai=0.55 | persuade_corpus | Distance learning | 735 words]
Interaction. Intercation is the way humans percieve the world around them. Collaborating among peers, communicating to conquor challenges and building up from past mistakes- that is how we learn.  However, in an increasingly digital society, it is quite easy to forget this fundemental aspect of leaning. Likewise, the rise of online or "distance" learning

## Step 9 — Is it too good to be true?

A test AUC of ~1.000 is exactly what the old, flawed version of this project reported, so it has to be challenged before it is believed. Three checks:

**9a. Echoes of human essays.** In `03_split` two "AI" essays turned out to be verbatim copies of the opening of a human essay. MinHash only catches a copy if it covers most of the human essay (Jaccard ≥ 0.7): a 400-word copy of a 700-word essay scores ~0.57 and slips through. If many such echoes existed, they'd be label noise and possible leakage. Search directly: AI essays whose first 20 words exactly match a human essay's opening, and how much of each AI essay is copied.

In [16]:
K = 20
opening = lambda t: ' '.join(t.split()[:K])
human_openings = {opening(t): i for i, t in data.loc[data['label'] == 0, 'text'].items()}

echoes = []
for i, r in data[data['label'] == 1].iterrows():
    h = human_openings.get(opening(r['text']))
    if h is None:
        continue
    a_words, h_words = r['text'].split(), data.loc[h, 'text'].split()
    n_same = next((k for k, (x, y) in enumerate(zip(a_words, h_words)) if x != y), min(len(a_words), len(h_words)))
    echoes.append({'ai_row': i, 'source': r['source'], 'ai_split': r['split'], 'human_split': data.loc[h, 'split'],
                   'copied_share': round(n_same / len(a_words), 3)})
echoes = pd.DataFrame(echoes)
print(f"AI essays sharing a 20-word opening with a human essay: {len(echoes)} of {(data['label'] == 1).sum()}")
echoes.sort_values('copied_share', ascending=False)

AI essays sharing a 20-word opening with a human essay: 14 of 17493


,ai_row,source,ai_split,human_split,copied_share
12,38107,mistral7binstruct_v2,train,train,0.998
1,28290,darragh_claude_v7,heldout_generator,train,0.087
8,33516,darragh_claude_v6,heldout_generator,train,0.086
10,35545,darragh_claude_v6,heldout_generator,train,0.074
3,28926,darragh_claude_v6,heldout_generator,train,0.073
5,29276,darragh_claude_v7,heldout_generator,train,0.071
9,35178,darragh_claude_v6,heldout_generator,train,0.071
7,32279,mistral7binstruct_v2,train,train,0.071
11,36188,darragh_claude_v6,heldout_generator,train,0.070
6,29888,darragh_claude_v6,heldout_generator,train,0.070


Only a handful, and nearly all copy just the first sentence (~7% of the essay): the generator was given the opening line as a starting point, then wrote the rest itself. That's legitimate AI text.

One essay (copied share ≈ 1.0, `mistral7binstruct_v2`) is a third echo of the kind dropped in `03`. It and its human source are both in `train`, so nothing crosses a split. It's one wrong label among ~9,700 AI training essays, so it can't explain the scores. Recorded in the data card.

**9b. Character-level shortcuts.** The whitespace artifact was neutralised in `03`. Check whether quote/apostrophe styles or anonymisation placeholders separate the classes (% of essays containing each):

In [17]:
patterns = {"straight '": "'", 'curly ’': '’', 'curly “ ”': '[“”]', 'any non-ASCII': r'[^\x00-\x7f]', 'Generic_ placeholder': 'Generic_'}
char_check = pd.DataFrame({k: data['text'].str.contains(v) for k, v in patterns.items()})
(char_check.groupby(data['label']).mean() * 100).round(1).rename(index={0: 'human', 1: 'AI'})

,straight ',curly ’,curly “ ”,any non-ASCII,Generic_ placeholder
label,,,,,
human,81.8,0.0,0.0,22.4,3.1
AI,64.2,4.9,2.0,11.1,1.0


No strong separator: curly quotes appear in only ~5% of AI essays, and `TfidfVectorizer`'s default tokenizer discards punctuation anyway. (The placeholder gap is small and gets checked again in Week 7's explainability work.)

**9c. Does the model need much data?** The sanity check in Step 4 already reached val AUC ≈ 0.997 from **2,000** essays. So the in-distribution task is easy: you don't need much data to separate grade 6–12 student writing (typos, plain vocabulary, "because", "would") from fluent LLM prose ("additionally", "in conclusion", "potential", "significant").

**Verdict:** the ~1.000 is real for *this dataset*, not leakage. But it measures "student essay vs 2023-era LLM essay", not "human vs AI" in general. The informative numbers are in the held-out slices.

## Step 10 — Save the results table

Saved to `results/04_baseline.csv` so later notebooks and the README rewrite can reuse the exact numbers.

In [18]:
import os
os.makedirs('../results', exist_ok=True)
table[cols].round(4).to_csv('../results/04_baseline.csv')
print(open('../results/04_baseline.csv').read())

,test_acc,test_precision,test_recall,test_f1,test_auc,heldout_prompt_f1,heldout_prompt_auc,heldout_gen_detection,heldout_gen_auc
always human,0.6544,0.0,0.0,0.0,0.5,0.0,0.5,0.0,0.5
length only (tree),0.6732,1.0,0.0545,0.1034,0.7189,0.0317,0.7259,0.0022,0.7502
prompt only,0.7298,0.6255,0.5438,0.5818,0.7633,0.0,0.5,0.2043,0.5129
TF-IDF + LogReg (C=10),0.9937,0.9975,0.9843,0.9908,0.9996,0.9131,0.9905,0.8971,0.9996
TF-IDF + LinearSVM (C=1),0.9966,1.0,0.9901,0.995,0.9998,0.9734,0.9971,0.9092,0.9997



## Key takeaways for the next notebooks

- **Baseline (TF-IDF + LinearSVM, C=1):** test F1 0.995 / AUC 1.000; held-out prompts F1 0.973 / AUC 0.997; held-out generators detection 90.9%, paired AUC 1.000. LogReg (C=10) is slightly behind: test F1 0.991, held-out prompts F1 0.913, generators 89.7%.
- **Reference points:** always-human gets accuracy 0.654 but F1 0; length-only AUC 0.72; prompt-only AUC 0.76 on test and exactly 0.5 on unseen prompts. TF-IDF is far beyond both surface signals.
- **No leakage found (Step 9).** The in-distribution task is simply easy on DAIGT-V2: 2,000 essays already give val AUC 0.997. It's really "student vs LLM", and that limitation goes in the report.
- **The real weakness is the threshold, not the ranking.** On unseen Claude essays the paired AUC is ~1.0, yet only ~85% are flagged at the default threshold: they still score above every human essay, just below 0.5. On the unseen prompts, "Summer projects" humans are flagged 4.8% of the time vs 0.1% on test, and only 80.8% of "Electoral college" AI essays are caught. **Calibration under distribution shift** is the problem to watch.
- **Topic leaks into the vocabulary:** "venus", "nasa", "car(s)", "earth" are among the strongest *human* terms, because those prompts are mostly human. The held-out-prompt slice is what exposes it.
- **What this means for Weeks 3–5:** every later model will also hit ~1.0 on `test`, so models must be compared on the held-out slices, on fixed-threshold metrics (detection rate, false-positive rate), and later on robustness: typo injection (Kaggle's hidden test added character noise for exactly this reason) and the Week 6 paraphrase attack.
- `results/04_baseline.csv` holds this table for the README rewrite.